# Pandas

## À quoi sert Pandas ?

**Pandas est LA bibliothèque pour manipuler des données tabulaires en Python.** En data science, **80% du temps** est consacré à préparer et nettoyer les données avant de les donner à un modèle — et la quasi-totalité de ce travail se fait avec pandas.

> **L'analogie à garder en tête :** pandas, c'est un **tableur Excel ou une base SQL accessible depuis Python**, avec la puissance d'un langage de programmation en bonus. Tout ce qu'on fait naturellement dans Excel (filtrer des lignes, calculer une colonne, faire une moyenne par catégorie...) devient une ligne de code pandas — mais **automatisable, reproductible et scalable** à des millions de lignes.

### Pourquoi pas un `for` Python classique ?

On *pourrait* charger un CSV et le parcourir avec une boucle `for`. Mais :
- ⚠️ **C'est lent** : Python itère ligne par ligne en code interprété. Pandas délègue à NumPy qui fait tout en C, **100 à 1000× plus rapide**.
- ⚠️ **C'est verbeux** : 20 lignes de code pour ce qui en prend 1 avec pandas.
- ⚠️ **C'est source d'erreurs** : gérer les valeurs manquantes, les types, les index à la main est un nid à bugs.

### Les deux structures fondamentales

Pandas repose sur deux objets, qu'on va voir en détail :

| Objet | À quoi ça ressemble |
|---|---|
| **`Series`** | Une colonne : un tableau 1D avec un index, un type et un nom |
| **`DataFrame`** | Un tableau 2D : une collection de `Series` partageant le même index |

Dans ce notebook on va couvrir : création, indexation (`loc`/`iloc`), filtrage, opérations, agrégations, valeurs manquantes, `groupby`... bref, **les gestes qu'on fait tous les jours en data science**.

In [1]:
import pandas as pd
import numpy as np

## DataFrames

Un `DataFrame` est la structure de données principale de pandas — c'est **un tableau 2D avec des colonnes nommées et un index de lignes**. Si tu ouvres un CSV dans Excel, ce que tu vois est très exactement un DataFrame.

**Points clés à retenir :**
- Chaque **colonne** est une `Series` avec son propre type (`int`, `float`, `str`, `date`...).
- Chaque **ligne** a un identifiant appelé **index** (par défaut : 0, 1, 2...).
- Toutes les colonnes partagent **le même index**, ce qui permet d'aligner les données.

On peut créer un DataFrame à partir de **plein de sources** : liste de listes, liste de dictionnaires, fichier CSV, base SQL, fichier Excel, API REST... On commence ici avec des petites structures Python pour comprendre.

In [2]:
# On peut créer un DataFrame à partir d'une liste de listes et une liste de colonnes

data = [
    ['Nissan', 'Stanza', 1991, 138, 4, 'MANUAL', 'sedan', 2000],
    ['Hyundai', 'Sonata', 2017, None, 4, 'AUTOMATIC', 'Sedan', 27150],
    ['Lotus', 'Elise', 2010, 218, 4, 'MANUAL', 'convertible', 54990],
    ['GMC', 'Acadia',  2017, 194, 4, 'AUTOMATIC', '4dr SUV', 34450],
    ['Nissan', 'Frontier', 2017, 261, 6, 'MANUAL', 'Pickup', 32340],
]

columns = [
    'Make', 'Model', 'Year', 'Engine HP', 'Engine Cylinders',
    'Transmission Type', 'Vehicle_Style', 'MSRP'
]

df = pd.DataFrame(data, columns=columns)

In [3]:
# on peut visualiser les premières lignes du DataFrame

df.head(3)

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990


In [4]:
# On peut aussi utiliser une liste de dictionnaires

data_dict = [
    {
        "Make": "Nissan",
        "Model": "Stanza",
        "Year": 1991,
        "Engine HP": 138.0,
        "Engine Cylinders": 4,
        "Transmission Type": "MANUAL",
        "Vehicle_Style": "sedan",
        "MSRP": 2000
    },
    {
        "Make": "Hyundai",
        "Model": "Sonata",
        "Year": 2017,
        "Engine HP": None,
        "Engine Cylinders": 4,
        "Transmission Type": "AUTOMATIC",
        "Vehicle_Style": "Sedan",
        "MSRP": 27150
    },
    {
        "Make": "Lotus",
        "Model": "Elise",
        "Year": 2010,
        "Engine HP": 218.0,
        "Engine Cylinders": 4,
        "Transmission Type": "MANUAL",
        "Vehicle_Style": "convertible",
        "MSRP": 54990
    },
    {
        "Make": "GMC",
        "Model": "Acadia",
        "Year": 2017,
        "Engine HP": 194.0,
        "Engine Cylinders": 4,
        "Transmission Type": "AUTOMATIC",
        "Vehicle_Style": "4dr SUV",
        "MSRP": 34450
    },
    {
        "Make": "Nissan",
        "Model": "Frontier",
        "Year": 2017,
        "Engine HP": 261.0,
        "Engine Cylinders": 6,
        "Transmission Type": "MANUAL",
        "Vehicle_Style": "Pickup",
        "MSRP": 32340
    }
]

df = pd.DataFrame(data_dict)

In [5]:
df.head(n=2)

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150


## Series

Une colonne dans un DataFrame est une **`Series`** : un tableau 1D de valeurs du **même type**, associé à un **index**. C'est très similaire à un tableau NumPy, mais avec deux atouts :

- Chaque valeur a un **label** (l'index), pas seulement une position. On peut donc accéder aux données par **nom** et pas seulement par numéro.
- Une `Series` connaît son **nom** (celui de la colonne), ce qui facilite le suivi et les opérations.

> **Mental model :** une `Series` = une colonne d'Excel + un ruban d'étiquettes sur le côté.

**Deux notations** pour accéder à une colonne depuis un DataFrame :
1. **Notation point** : `df.Make`. Élégante mais fragile — ne marche que si le nom de colonne est un identifiant Python valide (pas d'espaces, pas de caractères spéciaux, pas de conflit avec un attribut de `DataFrame` comme `.index` ou `.shape`).
2. **Notation crochets** : `df['Make']`. Toujours fonctionne, y compris `df['Engine HP']` (avec espace) ou `df[col_name]` (nom dans une variable). **À privilégier en pratique.**

In [6]:
# Notation ".". Nécessite que le nom de la colonne soit compatible avec celui
# d'un attribut python
df.Make

0     Nissan
1    Hyundai
2      Lotus
3        GMC
4     Nissan
Name: Make, dtype: object

In [7]:
# Notation chaîne de caractères.

df['Make']

0     Nissan
1    Hyundai
2      Lotus
3        GMC
4     Nissan
Name: Make, dtype: object

La notation chaîne de caractères est plus flexible

In [8]:
df['Engine HP']

0    138.0
1      NaN
2    218.0
3    194.0
4    261.0
Name: Engine HP, dtype: float64

In [9]:
col_name = 'Engine HP'
df[col_name]

0    138.0
1      NaN
2    218.0
3    194.0
4    261.0
Name: Engine HP, dtype: float64

## Manipulation des DataFrames

Maintenant qu'on sait créer et lire un DataFrame, voyons comment le **modifier** : sélectionner des colonnes, ajouter, remplacer, supprimer. Ce sont les opérations les plus fréquentes en préparation de données.

> **Règle importante :** par défaut, **beaucoup de méthodes pandas ne modifient pas l'objet d'origine** — elles retournent une *copie* modifiée. C'est un choix volontaire (permet le chaînage d'opérations, évite les bugs) mais ça surprend souvent au début. Pour modifier en place, il faut soit utiliser `inplace=True`, soit **réassigner** explicitement (`df = df.drop(...)`). On en reparlera plusieurs fois dans ce notebook.

In [10]:
# On peut sélectionner un sous-ensemble de colonnes. On obtient un nouveau DataFrame.

df[['Make', 'Model', 'MSRP']]

,Make,Model,MSRP
0,Nissan,Stanza,2000
1,Hyundai,Sonata,27150
2,Lotus,Elise,54990
3,GMC,Acadia,34450
4,Nissan,Frontier,32340


In [11]:
# On peut rajouter une colonne...

df['id'] = ['nis1', 'hyu1', 'lot2', 'gmc1', 'nis2']
df

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP,id
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000,nis1
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150,hyu1
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990,lot2
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450,gmc1
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340,nis2


In [12]:
# ... et la modifier
df['id'] = [1, 2, 3, 4, 5]
df

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP,id
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000,1
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150,2
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990,3
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450,4
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340,5


Pour supprimer une colonne

In [13]:
# Drop peut retourner une copie du DataFrame sans la colonne supprimée.
# Pandas parcourt l'axe 1 (axe des colonnes) pour chercher la colonne 'id' et la supprimer.

df2 = df.drop('id', axis=1)
df2

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340


In [14]:
# df est resté inchangé

df

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP,id
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000,1
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150,2
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990,3
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450,4
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340,5


In [15]:
# On peut supprimer la colonne in situ.

df.drop('id', axis=1, inplace= True)
df

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340


## Index

L'**index** est une des caractéristiques les plus importantes de pandas — et sans doute ce qui le distingue le plus d'un simple tableau NumPy. Les nombres (ou labels) à gauche d'un DataFrame ou d'une `Series` sont l'index.

### Pourquoi c'est important

- L'index permet d'**identifier** chaque ligne par un nom (ISBN, date, pseudo...) au lieu d'un simple numéro.
- Les opérations entre Series/DataFrames s'**alignent automatiquement** sur l'index. Si tu additionnes deux `Series` avec des index différents, pandas les aligne d'abord — pas d'erreurs silencieuses liées à l'ordre.
- L'index peut être **multi-niveaux** (MultiIndex) pour représenter des données hiérarchiques (ex : `(pays, année, mois)`).

> **À retenir :** quand pandas semble « magique », c'est souvent grâce à l'index. Et quand il te renvoie un résultat bizarre, c'est souvent **parce que** deux index ne s'alignent pas comme tu le pensais.

In [16]:
df = pd.DataFrame(data, columns=columns, index=['un', 'deux', 'trois', 'quatre', 'cinq'])
df

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
un,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
deux,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
trois,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
quatre,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450
cinq,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340


In [17]:
df.index

Index(['un', 'deux', 'trois', 'quatre', 'cinq'], dtype='object')

Par défaut, Pandas crée des index numériques auto-incrémentés et commençant à 0.

In [18]:
df = pd.DataFrame(data, columns=columns)
df

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340


In [19]:
df.index

RangeIndex(start=0, stop=5, step=1)

In [20]:
# Une Series possède aussi un index.

type(df.Make)

pandas.core.series.Series

In [21]:
# Un DataFrame possède deux index : un pour les lignes et un pour les colonnes.

df.columns

Index(['Make', 'Model', 'Year', 'Engine HP', 'Engine Cylinders',
       'Transmission Type', 'Vehicle_Style', 'MSRP'],
      dtype='object')

## Accès aux données

Il existe **trois façons** d'accéder aux données dans un DataFrame — et c'est probablement la source de confusion numéro 1 chez les débutants en pandas. Clarifions tout de suite :

| Méthode | Accès par | Exemple | Quand l'utiliser ? |
|---|---|---|---|
| **`df['col']`** | Nom de colonne | `df['Make']` | Sélection rapide d'une colonne |
| **`df.iloc[pos]`** | **Position** (numérique, comme en NumPy) | `df.iloc[0]` → 1ère ligne | Quand on raisonne en *positions* (ex : les 3 premières lignes) |
| **`df.loc[label]`** | **Label** d'index | `df.loc['un']` | Quand on raisonne en *noms* (ex : la ligne d'un client spécifique) |

> **Moyen mnémotechnique :**
> - `iloc` → *i* comme **integer** → position numérique.
> - `loc` → **label**, comme un nom.

Attention : **`df.iloc[0]` et `df.loc[0]` ne font pas la même chose !** Si l'index n'est pas 0, 1, 2..., les deux donneront des résultats différents. Piège classique à connaître.

In [22]:
df

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340


### Accès colonne

Celui que nous avons déjà utilisé

In [23]:
# Accès à la Series représentant la colonne 'Make'

df['Make']

0     Nissan
1    Hyundai
2      Lotus
3        GMC
4     Nissan
Name: Make, dtype: object

In [24]:
# Accès à l'élément d'index 2 de la Series représentant la colonne 'Make'

df['Make'][2]

'Lotus'

### iloc — accès par **position**

`iloc` fonctionne **exactement comme en NumPy** : on accède par numéro de ligne et de colonne, à partir de 0. Peu importe ce qu'il y a dans l'index — `iloc[0]` est *toujours* la première ligne.

- `df.iloc[0]` → 1ère ligne (en tant que `Series`)
- `df.iloc[0, 1]` → élément en 1ère ligne, 2ème colonne
- `df.iloc[[2, 3, 0]]` → les lignes 2, 3, 0 (dans cet ordre) — très utile pour réordonner !
- `df.iloc[:3]` → les 3 premières lignes (slicing classique Python)

In [25]:
# Accéder à la première ligne du DataFrame

df.iloc[0]

Make                 Nissan
Model                Stanza
Year                   1991
Engine HP             138.0
Engine Cylinders          4
Transmission Type    MANUAL
Vehicle_Style         sedan
MSRP                   2000
Name: 0, dtype: object

In [26]:
# Accéder à l'élément en première ligne et deuxième colonne

df.iloc[0, 1]

'Stanza'

In [27]:
# Accéder à un sous ensemble de lignes

df.iloc[[2, 3, 0]]

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000


On peut utiliser iloc pour mélanger les lignes d'un DataFrame

In [28]:
idx = np.arange(5)

# Mélanger l'index
np.random.seed(2)
np.random.shuffle(idx)
idx

array([2, 4, 1, 3, 0])

In [29]:
# Accéder aux lignes du DataFrame dans l'ordre déterminé par l'index.
# On obtient un nouveau DataFrame

df2 = df.iloc[idx]
df2

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000


In [30]:
# On peut accéder aux trois premières lignes du DataFrame mélangé,

df2.iloc[[0, 1, 2]]

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150


### loc — accès par **label**

`loc` accède aux données par **label d'index**, pas par position. Si tu as donné un index personnalisé (`'un'`, `'deux'`...), `loc` utilisera ces labels.

> **Cas particulier du piège :** quand l'index est numérique (0, 1, 2...), `loc[0]` et `iloc[0]` peuvent *sembler* équivalents — sauf quand l'ordre de l'index a été modifié (par exemple après un `shuffle`, un `sort_values`, ou un `train_test_split`). `iloc[0]` donnera alors la **première ligne dans l'ordre actuel**, tandis que `loc[0]` donnera **la ligne qui porte le label `0`**, qui peut être n'importe où dans le DataFrame.
>
> C'est pour ça que beaucoup de bugs subtils en pandas viennent d'un mélange `loc`/`iloc` après un tri ou un split.

In [31]:
df2.index

Index([2, 4, 1, 3, 0], dtype='int64')

On peut utiliser cet index pour accéder aux lignes avec loc

In [32]:
# On peut utiliser cet index pour accéder aux lignes grâce à loc

df2.loc[[0, 1]]

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150


In [33]:
# Avec iloc, on utilise la position

df2.iloc[[0, 1]]

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340


In [34]:
df2

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000


In [35]:
# On peut remplacer l'index par un index par défaut. Dans ce cas, l'ancien index est
# integré comme une nouvelle colonne

df2.reset_index(drop=False)

,index,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
1,4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340
2,1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
3,3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450
4,0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000


In [36]:
# Dans le cas où on ne veut pas conserver l'index comme colonne du DataFrame

df2.reset_index(drop=True)

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
1,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340
2,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450
4,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000


## Split d'un DataFrame

**Lien direct avec le ML :** c'est exactement ce qu'on fait pour séparer les données en **train / validation / test** avant d'entraîner un modèle (comme vu dans les notebooks précédents). En pratique, on utilise plutôt `train_test_split` de scikit-learn pour avoir le tirage aléatoire « gratuit », mais il est utile de savoir que derrière, ce sont juste des opérations `iloc` sur un DataFrame.

In [37]:
n_train = 3
n_val = 1
n_test = 1

In [38]:
df_train = df.iloc[:n_train]
df_val = df.iloc[n_train:n_train+n_val]
df_test = df.iloc[n_train+n_val:]

In [39]:
df_train

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990


In [40]:
df_val

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450


In [41]:
df_test

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340


In [42]:
df[['Make', 'Model', 'Year']]

,Make,Model,Year
0,Nissan,Stanza,1991
1,Hyundai,Sonata,2017
2,Lotus,Elise,2010
3,GMC,Acadia,2017
4,Nissan,Frontier,2017


In [43]:
df_train[['Make', 'Model', 'Year']]

,Make,Model,Year
0,Nissan,Stanza,1991
1,Hyundai,Sonata,2017
2,Lotus,Elise,2010


In [44]:
df_val[['Make', 'Model', 'Year']]

,Make,Model,Year
3,GMC,Acadia,2017


In [45]:
df_test[['Make', 'Model', 'Year']]

,Make,Model,Year
4,Nissan,Frontier,2017


## Opérations

### Le principe : la **vectorisation**

Pandas (comme NumPy) privilégie la **vectorisation** : on applique une opération à **toute une `Series` d'un coup**, sans écrire de boucle `for`. C'est à la fois :

- **Beaucoup plus concis** : `df['prix'] * 1.2` au lieu de boucler sur chaque ligne.
- **Beaucoup plus rapide** : les opérations sont déléguées à du code C/Fortran optimisé, des centaines de fois plus vite qu'une boucle Python.
- **Plus lisible** : le code dit *ce qu'on veut faire*, pas *comment le faire*.

> **Règle d'or en pandas :** *si tu écris un `for` sur un DataFrame, tu fais probablement quelque chose de mal.* Cherche d'abord la solution vectorisée.

### Opérations s'appliquant à chaque élément

Ce sont des opérations qui s'exécutent sur chaque élément d'une `Series` ou d'un DataFrame, **en parallèle**.


### Opérations arithmétiques

In [46]:
df['Engine HP']

0    138.0
1      NaN
2    218.0
3    194.0
4    261.0
Name: Engine HP, dtype: float64

In [47]:
# L'ensemble des opérations arithmétiques peuvent s'appliquer

df['Engine HP'] * 2

0    276.0
1      NaN
2    436.0
3    388.0
4    522.0
Name: Engine HP, dtype: float64

### Opérations logiques

In [48]:
df

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340


Les opérations logiques (`>`, `<`, `==`, `!=`, `>=`, `<=`) s'appliquent aussi **élément par élément**. Le résultat est une `Series` **booléenne** de même taille que l'original, avec `True` aux positions où la condition est vraie et `False` ailleurs.

> C'est la clé du **filtrage** qu'on voit juste après : on utilise ces masques booléens pour extraire seulement les lignes qui nous intéressent.

In [49]:
df['Year'] > 2000

0    False
1     True
2     True
3     True
4     True
Name: Year, dtype: bool

In [50]:
mask = df['Make'] == 'Nissan'
mask

0     True
1    False
2    False
3    False
4     True
Name: Make, dtype: bool

On peut combiner des expressions logiques avec les opérateurs **vectorisés** : `&` (ET), `|` (OU), `~` (NOT).

> ⚠️ **Piège classique :** on utilise `&`, `|`, `~` — **pas** `and`, `or`, `not`. Ces derniers sont les opérateurs Python « normaux » qui travaillent sur des scalaires, alors qu'on veut opérer élément par élément sur des Series. De plus, les parenthèses sont **obligatoires** autour de chaque condition à cause des priorités d'opérateurs :
>
> ```python
> df[(df['Year'] > 2010) & (df['Make'] == 'Nissan')]  # ✅ correct
> df[df['Year'] > 2010 & df['Make'] == 'Nissan']     # ❌ erreur
> ```

In [51]:
(df['Year'] > 2000) & (df['Make'] == 'Nissan')

0    False
1    False
2    False
3    False
4     True
dtype: bool

### Filtrage

Les tableaux booléens qu'on vient de créer servent à **filtrer** les lignes d'un DataFrame : `df[masque]` ne garde que les lignes où le masque vaut `True`.

> **Mental model :** `df[condition]` = « sélectionne-moi les lignes qui vérifient cette condition ». C'est l'équivalent direct d'un `WHERE` en SQL ou d'un filtre Excel — mais **enchaînable et programmable**.

C'est **l'opération la plus utilisée en data science** pour isoler des sous-populations, nettoyer des données ou répondre à des questions métier.

In [52]:
# Sélectionner les véhicules de marque Nissan

df[df['Make'] == 'Nissan']

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340


In [53]:
# qui équivaut à

df[[True, False, False, False, True]]

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340


In [54]:
# Véhicules immatriculés après 2010 et à boîte automatique

df[(df['Year'] > 2010) & (df['Transmission Type'] == 'AUTOMATIC')]

,Make,Model,Year,Engine HP,Engine Cylinders,Transmission Type,Vehicle_Style,MSRP
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450


### Opérations sur les chaînes de caractères

Pour traiter des colonnes de type `str`, pandas expose un **accesseur `.str`** qui regroupe toutes les opérations string classiques, vectorisées : `.str.lower()`, `.str.upper()`, `.str.replace()`, `.str.contains()`, `.str.split()`, etc.

> **Cas d'usage typique :** nettoyer des données sales.
>
> - Des noms de colonnes mal formattés (`"Engine HP"` → `"engine_hp"`) : une ligne suffit.
> - Des marques orthographiées de 10 façons différentes (`"BMW"`, `"bmw"`, `" BMW "`, `"B.M.W."`) : on normalise en une passe avec `.str.lower().str.strip().str.replace('.', '')`.
> - Extraire une info depuis une chaîne (ex : extraire le code postal d'une adresse) : `.str.extract(r'(\d{5})')`.
>
> **À retenir :** 90% du temps passé en data science, c'est du nettoyage. `.str` est votre ami.

In [55]:
df['Vehicle_Style']

0          sedan
1          Sedan
2    convertible
3        4dr SUV
4         Pickup
Name: Vehicle_Style, dtype: object

In [56]:
df['Vehicle_Style'].str.lower()

0          sedan
1          sedan
2    convertible
3        4dr suv
4         pickup
Name: Vehicle_Style, dtype: object

In [57]:
df['Vehicle_Style'].str.replace(' ', '_')

0          sedan
1          Sedan
2    convertible
3        4dr_SUV
4         Pickup
Name: Vehicle_Style, dtype: object

In [58]:
# On peut enchaîner les traitements, pouisque chaque fonction de str retourne
# un DataFrame ou une Series

df['Vehicle_Style'].str.lower().str.replace(' ', '_')

0          sedan
1          sedan
2    convertible
3        4dr_suv
4         pickup
Name: Vehicle_Style, dtype: object

In [59]:
# Cela s'applique aux Index et aux colonnes (qui sont elles-mêmes des Index)

df.columns

Index(['Make', 'Model', 'Year', 'Engine HP', 'Engine Cylinders',
       'Transmission Type', 'Vehicle_Style', 'MSRP'],
      dtype='object')

In [60]:
df.columns.str.lower().str.replace(' ', '_')

Index(['make', 'model', 'year', 'engine_hp', 'engine_cylinders',
       'transmission_type', 'vehicle_style', 'msrp'],
      dtype='object')

In [61]:
# On peut par exemple normaliser les noms de colonnes

df3 = df.copy()
df3.columns = df3.columns.str.lower().str.replace(' ', '_')

In [62]:
df3

,make,model,year,engine_hp,engine_cylinders,transmission_type,vehicle_style,msrp
0,Nissan,Stanza,1991,138.0,4,MANUAL,sedan,2000
1,Hyundai,Sonata,2017,NaN,4,AUTOMATIC,Sedan,27150
2,Lotus,Elise,2010,218.0,4,MANUAL,convertible,54990
3,GMC,Acadia,2017,194.0,4,AUTOMATIC,4dr SUV,34450
4,Nissan,Frontier,2017,261.0,6,MANUAL,Pickup,32340


On peut utiliser les mêmes opérations pour gérer des incohérences dans les colonnes.

Par exemple, on peut se servir de l'attribut `dtypes` pour selectionner les colonnes chaîne de caractères, qui en fait sont de type `object`.

In [63]:
df3.dtypes

make                  object
model                 object
year                   int64
engine_hp            float64
engine_cylinders       int64
transmission_type     object
vehicle_style         object
msrp                   int64
dtype: object

In [64]:
df3.dtypes.index

Index(['make', 'model', 'year', 'engine_hp', 'engine_cylinders',
       'transmission_type', 'vehicle_style', 'msrp'],
      dtype='object')

In [65]:
df3.dtypes == 'object'

make                  True
model                 True
year                 False
engine_hp            False
engine_cylinders     False
transmission_type     True
vehicle_style         True
msrp                 False
dtype: bool

In [66]:
# Series filtrée: on sélectionne les colonnes chaîne de caractères

df3.dtypes[df3.dtypes == 'object']

make                 object
model                object
transmission_type    object
vehicle_style        object
dtype: object

In [67]:
# Les noms des colonnes de type chaîne de caractères se trouvent dans l'index
# de la Series filtrée

df3.dtypes[df3.dtypes == 'object'].index

Index(['make', 'model', 'transmission_type', 'vehicle_style'], dtype='object')

In [68]:
# On peut maintenant itérer sur l'index obtenu pour parcourir les colonnes

string_columns = df3.dtypes[df3.dtypes == 'object'].index

df4 = df3.copy()
for col in string_columns:
    df4[col] = df4[col].str.lower().str.replace(' ', '_')

In [69]:
df4

,make,model,year,engine_hp,engine_cylinders,transmission_type,vehicle_style,msrp
0,nissan,stanza,1991,138.0,4,manual,sedan,2000
1,hyundai,sonata,2017,NaN,4,automatic,sedan,27150
2,lotus,elise,2010,218.0,4,manual,convertible,54990
3,gmc,acadia,2017,194.0,4,automatic,4dr_suv,34450
4,nissan,frontier,2017,261.0,6,manual,pickup,32340


## Opérations d'agrégation

Elles produisent **une valeur unique** (ou un petit résumé) à partir d'une colonne entière. Elles répondent à des questions comme :

- *« Quel est le prix moyen des voitures ? »* → `df['price'].mean()`
- *« Combien y a-t-il de marques différentes ? »* → `df['make'].nunique()`
- *« Quelle est la voiture la plus chère ? »* → `df['price'].max()`

Ce sont les **statistiques descriptives** du notebook `06-statistics-basics` — directement accessibles en une ligne de code pandas.

### Colonnes numériques

Pour les colonnes ou Series numériques, les opérations sont semblables à celles disponibles sur les tableaux NumPy.

In [70]:
df4.msrp

0     2000
1    27150
2    54990
3    34450
4    32340
Name: msrp, dtype: int64

In [71]:
# On peut calculer la moyenne, la somme, etc.

(
    df4.msrp.mean(),
    df4.msrp.median(),
    df4.msrp.sum(),
    df4.msrp.min(),
    df4.msrp.max(),
    df4.msrp.var(),
    df4.msrp.std()
)

(np.float64(30186.0),
 32340.0,
 np.int64(150930),
 2000,
 54990,
 360431930.0,
 18985.044903818372)

In [72]:
# On peut obtenir un ensemble de résultats grâce à l'opération `describe`,
# y compris les premier, deuxième et troisième quartiles

df4.msrp.describe()

count        5.000000
mean     30186.000000
std      18985.044904
min       2000.000000
25%      27150.000000
50%      32340.000000
75%      34450.000000
max      54990.000000
Name: msrp, dtype: float64

En appliquant une opération à un DataFrame, on obtient le résultat pour l'ensemble des colonnes numériques.

In [73]:
df4.mean(numeric_only=True)

year                 2010.40
engine_hp             202.75
engine_cylinders        4.40
msrp                30186.00
dtype: float64

In [74]:
df4.describe().round(2)

,year,engine_hp,engine_cylinders,msrp
count,5.00,4.00,5.00,5.00
mean,2010.40,202.75,4.40,30186.00
std,11.26,51.30,0.89,18985.04
min,1991.00,138.00,4.00,2000.00
25%,2010.00,180.00,4.00,27150.00
50%,2017.00,206.00,4.00,32340.00
75%,2017.00,228.75,4.00,34450.00
max,2017.00,261.00,6.00,54990.00


### Colonnes catégorielles

In [75]:
# Nombre de marques différentes

df4.make.nunique()

4

In [76]:
# Nombre de valeurs uniques pour chaque colonne

df4.nunique()

make                 4
model                5
year                 3
engine_hp            4
engine_cylinders     2
transmission_type    2
vehicle_style        4
msrp                 5
dtype: int64

In [77]:
# Tableau des marques (Chaque valeur différente n'apparaît qu'une fois)

df4.make.unique()

array(['nissan', 'hyundai', 'lotus', 'gmc'], dtype=object)

In [78]:
# Nombre de voitures par marque

df4.make.value_counts()

make
nissan     2
hyundai    1
lotus      1
gmc        1
Name: count, dtype: int64

## Traitement des valeurs manquantes

**Le problème :** en pratique, les données réelles sont **toujours** incomplètes. Un client n'a pas rempli sa date de naissance, un capteur a eu un bug, une colonne n'existe que dans la moitié des fichiers fusionnés... Les valeurs manquantes (notées `NaN` — *Not a Number*) sont partout.

### Pourquoi ça compte en data science

- **La plupart des modèles refusent les NaN** (`DecisionTreeClassifier`, `LinearRegression`, `SVM`...). On doit les traiter avant de fitter, sinon scikit-learn lève une erreur.
- **La façon dont on les traite influence les résultats**. Remplacer par 0, par la moyenne ou par la médiane donnera des modèles différents.

### Les stratégies courantes

| Stratégie | Code | Quand l'utiliser |
|---|---|---|
| **Supprimer les lignes** | `df.dropna()` | Quand il y a peu de NaN et qu'on peut se permettre de perdre des lignes |
| **Remplacer par 0** | `df.fillna(0)` | Quand « absence » a le sens de « zéro » (ex : nombre d'achats = 0) |
| **Remplacer par la moyenne/médiane** | `df.fillna(df.mean())` | Pour une variable numérique, stratégie simple et fréquente |
| **Remplacer par le mode** | `df.fillna(df.mode()[0])` | Pour une variable catégorielle |
| **Imputer plus finement** | `sklearn.impute.KNNImputer`, `IterativeImputer` | Quand la qualité compte et qu'on veut tenir compte des autres colonnes |

> ⚠️ **Attention au data leakage !** Si on impute avec la moyenne calculée sur **tout** le dataset, la moyenne du test « fuite » dans le train. La bonne pratique : calculer l'imputation **uniquement sur le train**, puis l'appliquer au test (via un `Pipeline` scikit-learn). Même principe que pour la normalisation.

In [79]:
df4

,make,model,year,engine_hp,engine_cylinders,transmission_type,vehicle_style,msrp
0,nissan,stanza,1991,138.0,4,manual,sedan,2000
1,hyundai,sonata,2017,NaN,4,automatic,sedan,27150
2,lotus,elise,2010,218.0,4,manual,convertible,54990
3,gmc,acadia,2017,194.0,4,automatic,4dr_suv,34450
4,nissan,frontier,2017,261.0,6,manual,pickup,32340


In [80]:
# DataFrame de booléens marquant les valeurs manquantes

df4.isnull()

,make,model,year,engine_hp,engine_cylinders,transmission_type,vehicle_style,msrp
0,False,False,False,False,False,False,False,False
1,False,False,False,True,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False


In [81]:
# Somme des valeurs à True (True vaut 1), True correspondant
# à une valeur manquante. Appliquée au DataFrame, on obtient le nombre de valeurs
# manquantes pour chaque colonne

df4.isnull().sum()

make                 0
model                0
year                 0
engine_hp            1
engine_cylinders     0
transmission_type    0
vehicle_style        0
msrp                 0
dtype: int64

In [82]:
# On peut aussi obtenir le nombre de valeurs manquantes pour chaque ligne
# (le long de l'axe des colonnes, axis=1). Par défaut, les opérations se font sur l'axe
# des lignes, axis=0

df4.isnull().sum(axis=1)

0    0
1    1
2    0
3    0
4    0
dtype: int64

In [83]:
# On peut obtenir des informations synthétiques sur l'ensemble des colonnes du DataFrame avec la fonction info()
# Et constater l'existence d'une valeur nulle dans la colonne engine_hp

df4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   make               5 non-null      object 
 1   model              5 non-null      object 
 2   year               5 non-null      int64  
 3   engine_hp          4 non-null      float64
 4   engine_cylinders   5 non-null      int64  
 5   transmission_type  5 non-null      object 
 6   vehicle_style      5 non-null      object 
 7   msrp               5 non-null      int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 452.0+ bytes


In [84]:
df4.engine_hp.isnull()

0    False
1     True
2    False
3    False
4    False
Name: engine_hp, dtype: bool

In [85]:
# On peut remplacer les valeurs nulles par 0

df4.engine_hp.fillna(0)

0    138.0
1      0.0
2    218.0
3    194.0
4    261.0
Name: engine_hp, dtype: float64

In [86]:
# ou par une moyenne

df4.engine_hp.fillna(df4.engine_hp.mean())

0    138.00
1    202.75
2    218.00
3    194.00
4    261.00
Name: engine_hp, dtype: float64

In [87]:
# Comme la plupart des opérations de mutation, fillna ne modifie pas l'objet auquel il
# s'applique

df4

,make,model,year,engine_hp,engine_cylinders,transmission_type,vehicle_style,msrp
0,nissan,stanza,1991,138.0,4,manual,sedan,2000
1,hyundai,sonata,2017,NaN,4,automatic,sedan,27150
2,lotus,elise,2010,218.0,4,manual,convertible,54990
3,gmc,acadia,2017,194.0,4,automatic,4dr_suv,34450
4,nissan,frontier,2017,261.0,6,manual,pickup,32340


In [88]:
# Pour que la modification soit effective, il faut faire l'affectation explicitement

df5 = df4.copy()

# Modifier la colonne engine_hp
df5.engine_hp = df5.engine_hp.fillna(df5.engine_hp.mean())
df5

,make,model,year,engine_hp,engine_cylinders,transmission_type,vehicle_style,msrp
0,nissan,stanza,1991,138.00,4,manual,sedan,2000
1,hyundai,sonata,2017,202.75,4,automatic,sedan,27150
2,lotus,elise,2010,218.00,4,manual,convertible,54990
3,gmc,acadia,2017,194.00,4,automatic,4dr_suv,34450
4,nissan,frontier,2017,261.00,6,manual,pickup,32340


## Tri

Trier un DataFrame est une opération courante : pour afficher les top-N résultats, pour trouver les valeurs extrêmes, ou simplement pour rendre une exploration plus lisible.

> **À savoir :** `sort_values` **retourne une copie triée** (comme la plupart des méthodes pandas). L'original reste inchangé. Pour modifier en place : `df.sort_values(..., inplace=True)`, ou réassigner.

In [89]:
df5

,make,model,year,engine_hp,engine_cylinders,transmission_type,vehicle_style,msrp
0,nissan,stanza,1991,138.00,4,manual,sedan,2000
1,hyundai,sonata,2017,202.75,4,automatic,sedan,27150
2,lotus,elise,2010,218.00,4,manual,convertible,54990
3,gmc,acadia,2017,194.00,4,automatic,4dr_suv,34450
4,nissan,frontier,2017,261.00,6,manual,pickup,32340


In [90]:
# Trier selon les valeurs de msrp ascendantes

df5.sort_values(by='msrp')

,make,model,year,engine_hp,engine_cylinders,transmission_type,vehicle_style,msrp
0,nissan,stanza,1991,138.00,4,manual,sedan,2000
1,hyundai,sonata,2017,202.75,4,automatic,sedan,27150
4,nissan,frontier,2017,261.00,6,manual,pickup,32340
3,gmc,acadia,2017,194.00,4,automatic,4dr_suv,34450
2,lotus,elise,2010,218.00,4,manual,convertible,54990


In [91]:
# Trier selon les valeurs de msrp descendantes

df5.sort_values(by='msrp', ascending=False)

,make,model,year,engine_hp,engine_cylinders,transmission_type,vehicle_style,msrp
2,lotus,elise,2010,218.00,4,manual,convertible,54990
3,gmc,acadia,2017,194.00,4,automatic,4dr_suv,34450
4,nissan,frontier,2017,261.00,6,manual,pickup,32340
1,hyundai,sonata,2017,202.75,4,automatic,sedan,27150
0,nissan,stanza,1991,138.00,4,manual,sedan,2000


## Groupby

**L'opération la plus puissante de pandas — et probablement la plus utilisée en data analysis.**

### Le mental model : Split-Apply-Combine

`groupby` suit un schéma en 3 étapes qui porte un nom connu en data engineering, **Split-Apply-Combine** :

1. **Split** : on découpe le DataFrame en sous-groupes selon les valeurs d'une (ou plusieurs) colonne(s). Par exemple « toutes les lignes dont `transmission_type == 'AUTOMATIC'` », « ...`MANUAL'` », etc.
2. **Apply** : on applique une opération à chaque sous-groupe (moyenne, somme, count, fonction custom...).
3. **Combine** : on recolle les résultats dans un seul DataFrame ou Series.

### À quoi ça sert concrètement

Presque toutes les questions analytiques du type *« par X, quelle est la Y ? »* se répondent en un `groupby` :

- *« Prix moyen par type de transmission ? »* → `df.groupby('transmission').msrp.mean()`
- *« Nombre de ventes par jour ? »* → `df.groupby('date').size()`
- *« Meilleur vendeur par région ? »* → `df.groupby('region')['ventes'].idxmax()`
- *« Taux de conversion par canal marketing ? »* → `df.groupby('canal')['converti'].mean()`

C'est l'équivalent direct du `GROUP BY` en SQL, mais **en plus expressif** (on peut appliquer n'importe quelle fonction Python, pas juste les agrégations standards).

On peut vouloir effectuer des opérations d'aggrégation non pas sur l'ensemble d'un DataFrame mais sur chaque groupement selon les valeurs d'une ou plusieurs colonnes, comme par exemple le prix moyen par type de transmission.

EN SQL, on aurait quelque chose du genre:

    SELECT
        tranmission_type,
        AVG(msrp)
    FROM
        cars
    GROUP BY
        transmission_type`

In [92]:
# prix moyen par type de transmission

df5.groupby('transmission_type').msrp.mean()

transmission_type
automatic    30800.000000
manual       29776.666667
Name: msrp, dtype: float64

In [93]:
# On peut regrouper selon deux colonnes:

df5.groupby(by=['transmission_type', 'engine_cylinders']).msrp.mean()

transmission_type  engine_cylinders
automatic          4                   30800.0
manual             4                   28495.0
                   6                   32340.0
Name: msrp, dtype: float64

In [94]:
# On peut appliquer plusieurs aggrégations à un groupby, et obtenir le résultat sous
# la forme d'un DataFrame.
# Pour chaque type de transmission, on voudrait connaître le prix moyen
# et le nombre de voitures

df_group = df5.groupby('transmission_type').msrp.agg(['mean', 'count'])
df_group

,mean,count
transmission_type,,
automatic,30800.000000,2
manual,29776.666667,3


In [95]:
# ... et comparer à la moyenne globale

df_group['mean'] - df5.msrp.mean()

transmission_type
automatic    614.000000
manual      -409.333333
Name: mean, dtype: float64

## Series et tableaux NumPy

**Une `Series` pandas est en réalité un tableau NumPy avec un index et un nom en plus.** Tout ce que NumPy sait faire sur un tableau 1D, on peut le faire sur une Series :

- Fonctions mathématiques : `np.log`, `np.sqrt`, `np.exp`, `np.log1p`...
- Broadcasting entre Series et scalaires ou autres tableaux.
- Accès au tableau NumPy sous-jacent via `.values` (ou `.to_numpy()`, plus récent).

> **Pourquoi c'est important ?**
> Beaucoup de bibliothèques scientifiques (scikit-learn, scipy, matplotlib, statsmodels...) attendent des **tableaux NumPy** en entrée. Heureusement, pandas et NumPy s'interconnectent naturellement : dans 99% des cas, tu peux passer une `Series` ou un `DataFrame` directement à scikit-learn, et il saura s'en servir. Quand tu as besoin d'un tableau NumPy explicite (par exemple pour faire un calcul très optimisé), `.values` te le donne.

In [96]:
df5.msrp

0     2000
1    27150
2    54990
3    34450
4    32340
Name: msrp, dtype: int64

In [97]:
np.log1p(df5.msrp)

0     7.601402
1    10.209169
2    10.914925
3    10.447293
4    10.384091
Name: msrp, dtype: float64

In [98]:
# On peut obtenir le tableau NumPy sous-jacent

df5.msrp.values

array([ 2000, 27150, 54990, 34450, 32340])

In [99]:
np.log1p(df5.msrp.values)

array([ 7.60140233, 10.20916916, 10.91492481, 10.4472933 , 10.38409105])

## Convertir en dictionnaires python

In [100]:
df5.to_dict()

{'make': {0: 'nissan', 1: 'hyundai', 2: 'lotus', 3: 'gmc', 4: 'nissan'},
 'model': {0: 'stanza', 1: 'sonata', 2: 'elise', 3: 'acadia', 4: 'frontier'},
 'year': {0: 1991, 1: 2017, 2: 2010, 3: 2017, 4: 2017},
 'engine_hp': {0: 138.0, 1: 202.75, 2: 218.0, 3: 194.0, 4: 261.0},
 'engine_cylinders': {0: 4, 1: 4, 2: 4, 3: 4, 4: 6},
 'transmission_type': {0: 'manual',
  1: 'automatic',
  2: 'manual',
  3: 'automatic',
  4: 'manual'},
 'vehicle_style': {0: 'sedan',
  1: 'sedan',
  2: 'convertible',
  3: '4dr_suv',
  4: 'pickup'},
 'msrp': {0: 2000, 1: 27150, 2: 54990, 3: 34450, 4: 32340}}

In [101]:
# Plusieurs formats sont disponibles. Voir la doc de to_dict

df5.to_dict(orient='list')

{'make': ['nissan', 'hyundai', 'lotus', 'gmc', 'nissan'],
 'model': ['stanza', 'sonata', 'elise', 'acadia', 'frontier'],
 'year': [1991, 2017, 2010, 2017, 2017],
 'engine_hp': [138.0, 202.75, 218.0, 194.0, 261.0],
 'engine_cylinders': [4, 4, 4, 4, 6],
 'transmission_type': ['manual', 'automatic', 'manual', 'automatic', 'manual'],
 'vehicle_style': ['sedan', 'sedan', 'convertible', '4dr_suv', 'pickup'],
 'msrp': [2000, 27150, 54990, 34450, 32340]}

## 🎯 Pandas en pratique — ce qu'il faut retenir

### Les réflexes à avoir

**Pour explorer un nouveau dataset, la routine « 5 lignes » :**
```python
df.shape          # (lignes, colonnes)
df.head()         # les 5 premières lignes pour voir la tête
df.dtypes         # types de chaque colonne (détecte les erreurs de parsing)
df.isnull().sum() # nombre de NaN par colonne (où sont les trous ?)
df.describe()     # stats descriptives des colonnes numériques
```

C'est **toujours** la première chose à faire sur un jeu de données inconnu. Si l'un de ces 5 commandes renvoie quelque chose de bizarre, tu as identifié ton premier problème de qualité des données.

### Les patterns de base à maîtriser

| Besoin | Code |
|---|---|
| Filtrer des lignes | `df[df['col'] > 10]` |
| Sélectionner des colonnes | `df[['col1', 'col2']]` |
| Ajouter une colonne calculée | `df['ratio'] = df['a'] / df['b']` |
| Remplir les manquants | `df['col'] = df['col'].fillna(df['col'].median())` |
| Agréger par groupe | `df.groupby('cat')['val'].mean()` |
| Trier | `df.sort_values('col', ascending=False)` |
| Fusionner deux DataFrames | `df1.merge(df2, on='id')` |
| Pivoter (large → long) | `df.melt(id_vars='id')` |

### Les pièges à connaître

- ⚠️ **Chaînage d'assignation** (`SettingWithCopyWarning`) : faire `df[df['a'] > 0]['b'] = 42` ne modifie **pas** `df` de façon fiable. Utiliser `df.loc[df['a'] > 0, 'b'] = 42` à la place.
- ⚠️ **`inplace=True` est souvent un piège** : la plupart des experts recommandent de **réassigner** (`df = df.drop(...)`) plutôt que d'utiliser `inplace=True`. Plus clair, plus chaînable, et souvent plus rapide.
- ⚠️ **Boucles `for`** : si tu itères sur un DataFrame avec un `for`, arrête-toi et cherche d'abord la solution vectorisée ou avec `apply` / `groupby`. C'est 99% du temps possible, et 100 à 1000× plus rapide.
- ⚠️ **`==` vs `is`** : `df['col'] == None` ne détecte **pas** les NaN ! Utiliser `df['col'].isnull()`.
- ⚠️ **Index oublié** : après un `groupby`, un `sort`, un `reset_index`... toujours vérifier que l'index est bien celui qu'on attend, surtout avant un `merge` ou un `concat`.

### Pour aller plus loin

- **Pandas method chaining** : enchaîner les opérations (`.pipe`, `.assign`, `.query`) pour un style plus lisible.
- **Polars** : alternative moderne à pandas, **beaucoup plus rapide** sur les gros datasets, avec une API inspirée de pandas et SQL. À connaître pour 2024+.
- **DuckDB** : base SQL in-process qui travaille directement sur des DataFrames pandas — idéal pour des requêtes complexes type SQL sur des données locales.

### Le mot de la fin

> **Pandas est un outil d'exploration, pas seulement de préparation.** Passe du temps à « parler » à tes données avant de modéliser : un bon data scientist connaît ses données *avant* de connaître ses modèles. Les 5 premières minutes à faire `.describe()`, `.value_counts()`, `.groupby()` sur un dataset t'éviteront souvent des heures de debug plus tard.